<a href="https://colab.research.google.com/github/nafissa-create/swahili-sentiment-afroxlmr/blob/main/swahili_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1-Install and load the data

!pip install ... installs the libraries. datasets downloads data, transformers has the model, accelerate helps with training, and scikit-learn is for the baseline and the scores. The ! means "run this in the terminal, not in Python".
load_dataset("HausaNLP/AfriSenti-Twitter", "swa", ...) downloads the Swahili part of AfriSenti. "swa" is the language code. trust_remote_code=True is needed because this dataset ships with a loading script.
The result ds has three parts: train (the model learns from it), validation (we use it to check progress and pick the best epoch), and test (kept aside until the end for the honest final score).
to_df turns each part into a pandas table with two columns: tweet and label. The if hasattr(...) line handles the case where labels arrive as numbers, by converting them back to words.

In [2]:
!pip install -q "datasets<4.0" transformers accelerate scikit-learn

from datasets import load_dataset
import pandas as pd

ds = load_dataset("HausaNLP/AfriSenti-Twitter", "swa", trust_remote_code=True)
print(ds)

def to_df(split):
    d = ds[split].to_pandas()
    feat = ds[split].features["label"]
    if hasattr(feat, "names"):          # in case labels are stored as numbers
        d["label"] = d["label"].map(lambda i: feat.names[i])
    return d

train_df, val_df, test_df = to_df("train"), to_df("validation"), to_df("test")
print(len(train_df), len(val_df), len(test_df))
train_df.head()

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.3.0 which is incompatible.


README.md:   0%|          | 0.00/8.44k [00:00<?, ?B/s]

AfriSenti-Twitter.py:   0%|          | 0.00/5.66k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

----------------------------------------------------------------------------------------------------
                                               tweet     label
0  Kwani tanesco wanakataga umeme makusudinadhani...  negative
1  cjawahi kuona content yoyote zaidi ya kuwa ana...  negative
2  Bomu lililokuwa limetegwa ndani ya gari likiwa...  negative
3  Kuna video inasambaa mitandaoni jamaa amemfuma...  negative
4  Viwavijeshi wanapita katika hatua kuu 6 za uku...  negative
----------------------------------------------------------------------------------------------------


Generating validation split: 0 examples [00:00, ? examples/s]

----------------------------------------------------------------------------------------------------
                                               tweet     label
0  Leo nimepata kitambulisho changu cha taifa Asa...  positive
1  Asante na tafadhali subiri tutawasiliana nawe ...  positive
2  Takwimu zinaonyesha katika kila watoto 100 27 ...   neutral
3  Kwa hiyo kama ilivyokuja kwenye aya hiyo ni ju...   neutral
4  Sio watu wazuri awa Nakutana nao FA Cup nawafa...  negative
----------------------------------------------------------------------------------------------------


Generating test split: 0 examples [00:00, ? examples/s]

----------------------------------------------------------------------------------------------------
                                               tweet     label
0     Ulitaka kujua siasa hzi za democrazy nawaom...  positive
1  Chama chenye sera kali za mrengo wa kulia amba...   neutral
2     Mwaka mmoja wa Rais Magufuli na mkwamo wa e...  negative
3     Umeondoka kimya kimya mzee wangu Sijapenda ...   neutral
4  Watu wengine hawatakupenda hata ufanye nini na...  positive
----------------------------------------------------------------------------------------------------
DatasetDict({
    train: Dataset({
        features: ['tweet', 'label'],
        num_rows: 1810
    })
    validation: Dataset({
        features: ['tweet', 'label'],
        num_rows: 453
    })
    test: Dataset({
        features: ['tweet', 'label'],
        num_rows: 748
    })
})
1810 453 748


,tweet,label
0,Kwani tanesco wanakataga umeme makusudinadhani...,negative
1,cjawahi kuona content yoyote zaidi ya kuwa ana...,negative
2,Bomu lililokuwa limetegwa ndani ya gari likiwa...,negative
3,Kuna video inasambaa mitandaoni jamaa amemfuma...,negative
4,Viwavijeshi wanapita katika hatua kuu 6 za uku...,negative


2-Inspect and export to CSV

It saves each split as a .csv file. That's your "dataset and preparation documents" deliverable.
It counts the labels in each split, finds duplicate and empty tweets, and describes the tweet lengths. Never train on data you haven't looked at. These counts tell us whether the classes are balanced and whether cleaning is needed.

In [3]:
import os
os.makedirs("data", exist_ok=True)
for name, d in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    d.to_csv(f"data/afrisenti_swa_{name}.csv", index=False)
    print(name, d["label"].value_counts().to_dict())

print("Labels found:", sorted(train_df["label"].unique()))
print("Duplicates in train:", train_df.duplicated(subset="tweet").sum())
print("Empty tweets:", (train_df["tweet"].str.strip() == "").sum())
print("Tweet length in words:", train_df["tweet"].str.split().str.len().describe())

train {'neutral': 1072, 'positive': 547, 'negative': 191}
validation {'neutral': 268, 'positive': 137, 'negative': 48}
test {'neutral': 444, 'positive': 224, 'negative': 80}
Labels found: ['negative', 'neutral', 'positive']
Duplicates in train: 8
Empty tweets: 0
Tweet length in words: count    1810.000000
mean       16.908840
std         8.239772
min         2.000000
25%        11.000000
50%        16.000000
75%        20.000000
max        45.000000
Name: tweet, dtype: float64


In [5]:
dups = train_df[train_df.duplicated(subset="tweet", keep=False)].sort_values("tweet")
print(dups)
print("train/test overlap:", len(set(train_df.tweet) & set(test_df.tweet)))
print("train/val overlap:", len(set(train_df.tweet) & set(val_df.tweet)))
# 1. tweets that appear with more than one label are unreliable: drop all copies
n_labels = train_df.groupby("tweet")["label"].nunique()
conflict_tweets = set(n_labels[n_labels > 1].index)
print("conflicting tweets:", len(conflict_tweets))

# 2. keep one copy of the harmless duplicates
train_clean = (train_df[~train_df["tweet"].isin(conflict_tweets)]
               .drop_duplicates(subset="tweet").reset_index(drop=True))

# 3. remove validation tweets that also appear in the original train set
val_clean = val_df[~val_df["tweet"].isin(set(train_df["tweet"]))].reset_index(drop=True)

# test set stays untouched so the final score is comparable to other AfriSenti work
train_df, val_df = train_clean, val_clean
print(len(train_df), len(val_df), len(test_df))
print(train_df["label"].value_counts().to_dict())

train_df.to_csv("data/afrisenti_swa_train_clean.csv", index=False)
val_df.to_csv("data/afrisenti_swa_validation_clean.csv", index=False)

                                                  tweet     label
1334  HABARI NJEMA KWA MASHIBIKI ZANGU Kwa mujibu wa...  positive
1339  HABARI NJEMA KWA MASHIBIKI ZANGU Kwa mujibu wa...  positive
340   Habari Ahsante kwa uvumilivu wako pole sana kw...   neutral
485   Habari Ahsante kwa uvumilivu wako pole sana kw...   neutral
404   Habari pole sana kwa changamoto tunaomba namba...   neutral
1041  Habari pole sana kwa changamoto tunaomba namba...   neutral
322    Habari tunaomba namba yako DM kwa msaada zaidiKA   neutral
1308   Habari tunaomba namba yako DM kwa msaada zaidiKA  positive
1577   Habari tunaomba namba yako DM kwa msaada zaidiKA  positive
1382  Ishi na watu vizuri watendee wema MUNGU ndiye ...  positive
1542  Ishi na watu vizuri watendee wema MUNGU ndiye ...  positive
1241       Tunafunga sana kumi na nusu jionikaribu sana   neutral
1749       Tunafunga sana kumi na nusu jionikaribu sana  positive
1222  Upo eneo gani mteja tunaomba details vizuri il...   neutral
1430  Upo 

3- Simple baseline

TfidfVectorizer turns each tweet into numbers by counting words and word pairs, giving rarer words more weight. It's the same idea as CountVectorizer from class.
LogisticRegression learns from those numbers. class_weight="balanced" makes mistakes on rare classes count more.
Why a baseline? A score means little on its own. If fine-tuning gets 0.60 macro F1 and this simple method gets 0.58, the story is different than if the simple method gets 0.40.

Why macro F1? Imagine 70% of tweets are neutral. A useless model that always says "neutral" scores 70% accuracy and looks good. F1 for one class balances two questions: of the tweets I labelled positive, how many truly were (precision), and of all the truly positive tweets, how many did I find (recall). Macro F1 computes F1 for each class and averages them with equal weight, so a model that ignores a small class gets punished. Look at the label counts from Cell 2 to see whether this matters here.

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
Xtr = vec.fit_transform(train_df["tweet"])
base = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, train_df["label"])

pred = base.predict(vec.transform(test_df["tweet"]))
print("Baseline macro F1 on test:", f1_score(test_df["label"], pred, average="macro"))
print(classification_report(test_df["label"], pred))

Baseline macro F1 on test: 0.46314140227183703
              precision    recall  f1-score   support

    negative       0.27      0.35      0.30        80
     neutral       0.66      0.61      0.63       444
    positive       0.44      0.46      0.45       224

    accuracy                           0.54       748
   macro avg       0.46      0.47      0.46       748
weighted avg       0.55      0.54      0.54       748



4-Fine-tune afro-xlmr-base

MODEL = "Davlan/afro-xlmr-base" is the base model.
labels and the two dictionaries map the words (negative, neutral, positive) to numbers 0, 1, 2 and back. The model only works with numbers.
The tokenizer cuts each tweet into pieces and turns them into numbers. truncation=True, max_length=128 cuts very long tweets at 128 pieces. make(df) applies it to a whole table.
AutoModelForSequenceClassification loads the base model and adds a small new layer on top that outputs 3 scores, one per class. That layer starts untrained, and fine-tuning teaches it.
metrics computes macro F1 and accuracy after each epoch.
TrainingArguments holds the settings:
num_train_epochs=5: the model sees the whole training set 5 times.
per_device_train_batch_size=16: it learns from 16 tweets at a time.
learning_rate=2e-5: the size of each adjustment. It's small on purpose, so we nudge the pretrained model and don't destroy what it knows.
eval_strategy="epoch": check the validation set after each epoch.
load_best_model_at_end=True with metric_for_best_model="macro_f1": keep the epoch that scored best on validation, not simply the last one.
fp16=True: a faster number format on the GPU.
seed=42: makes the run repeatable.
Trainer(...) bundles everything. DataCollatorWithPadding pads the tweets in each batch to the same length. trainer.train() runs the loop: for each batch, predict, measure the error (cross-entropy loss), and adjust the weights to reduce it.


In [7]:
import numpy as np, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding)
from sklearn.metrics import f1_score, accuracy_score

MODEL = "Davlan/afro-xlmr-base"
labels = ["negative", "neutral", "positive"]
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

tok = AutoTokenizer.from_pretrained(MODEL)

def make(df):
    d = Dataset.from_dict({"text": df["tweet"].tolist(),
                           "label": [label2id[l] for l in df["label"]]})
    d = d.map(lambda b: tok(b["text"], truncation=True, max_length=128), batched=True)
    return d.remove_columns(["text"])

train_ds, val_ds, test_ds = make(train_df), make(val_df), make(test_df)

def metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro"),
            "accuracy": accuracy_score(p.label_ids, preds)}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL, num_labels=3, id2label=id2label, label2id=label2id)

args = TrainingArguments(
    output_dir="out", num_train_epochs=5,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, weight_decay=0.01,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="macro_f1",
    save_total_limit=1, fp16=torch.cuda.is_available(),
    report_to="none", seed=42)

trainer = Trainer(model=model, args=args, train_dataset=train_ds,
                  eval_dataset=val_ds, data_collator=DataCollatorWithPadding(tok),
                  compute_metrics=metrics)
trainer.train()

config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

... (10 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=565, training_loss=0.6415522111200653, metrics={'train_runtime': 478.5046, 'train_samples_per_second': 18.798, 'train_steps_per_second': 1.181, 'total_flos': 265729135103460.0, 'train_loss': 0.6415522111200653, 'epoch': 5.0})

5-Evaluate on the test set

This is the final exam. trainer.predict(test_ds) predicts on tweets the model has never seen, and then we print macro F1, a per-class report, and a confusion matrix. The confusion matrix shows which classes get mixed up, for example neutral tweets labelled positive. It's the starting point for your error analysis.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

out = trainer.predict(test_ds)
preds = np.argmax(out.predictions, axis=1)
print("macro F1:", f1_score(out.label_ids, preds, average="macro"))
print(classification_report(out.label_ids, preds, target_names=labels))
print(confusion_matrix(out.label_ids, preds))

6-Save the model and try it on your own sentences

save_model and save_pretrained write the fine-tuned model and tokenizer to a folder. This is your "fine-tuned model" deliverable.
pipeline("text-classification", ...) wraps the model so you can pass in a sentence and get a label and confidence. The three example sentences are simple Swahili, and the English meanings are in the comments. They are a sanity check, not a proper test.

In [ ]:
trainer.save_model("afro-xlmr-swahili-sentiment")
tok.save_pretrained("afro-xlmr-swahili-sentiment")

from transformers import pipeline
clf = pipeline("text-classification", model="afro-xlmr-swahili-sentiment",
               device=0 if torch.cuda.is_available() else -1)
print(clf("Huduma ilikuwa nzuri sana, asante!"))        # "The service was very good, thanks!"
print(clf("Hii ni mbaya sana, sijafurahi"))              # "This is very bad, I am not happy"
print(clf("Mkutano utafanyika kesho saa tatu"))          # "The meeting will be held tomorrow at nine"

7- Copy the results to Google Drive

It copies your files to Google Drive, since Colab's own storage disappears when the session ends.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!cp -r data afro-xlmr-swahili-sentiment /content/drive/MyDrive/